# AG_PRAXIS NB06 — Sequence Model

Every row in this dataset is a summary of a short stretch of traffic: how fast packets
arrived, how big they were, which flags were set, how much the sizes varied. A model
given one row at a time has to decide what kind of traffic it is looking at from that
summary alone. This notebook trains a model that is given fifty consecutive rows
instead, and reports what that changes.

The model is two pieces joined together. The first is the convolutional feature
extractor from the published network this project reproduces, taken exactly as it is
published and applied to each of the fifty records in turn, which turns each record into
a short vector. The second is a single LSTM layer that reads those fifty vectors in
order, with a dense softmax over the nineteen classes on the end. The extractor is not
altered in any way. Reading across the window is the one thing that is added, so if the
score moves there is one candidate for why.

One run. Seed 42, windows of fifty records at a stride of twenty-five, all forty-four
columns the files leave once the one constant column is dropped, ten epochs at batch 32,
and the split that holds recording sessions apart. Nothing is searched over and nothing
is tuned, because a search would put more than one difference between this run and the
one it is compared against.

Results are reported as overall accuracy, weighted F1 and macro F1 together, and then
per class for all nineteen. On a corpus where the largest class holds more than two
thousand times the rows of the smallest, an average taken over rows and an average taken
over classes answer different questions, and the distance between them is where the rare
classes show up. Some classes leave very few windows at this length, and the metrics file
says which and how few in a field of its own rather than leaving it to be noticed.

The data sits on Drive and the code sits in the repository, so the first block mounts one
and clones the other, and records the commit it is running from.

In [1]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"
NOTEBOOK = "AG_PRAXIS_NB06_sequence_model.ipynb"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Mounted at /content/drive
colab     : True
repo root : /content/repo
git sha   : 20d4709 on main
run date  : 2026-08-07


The parameters come from `config/base.yaml`: the seed, the window and stride, the batch
size, the number of epochs, and where things are read from and written to. What the data
looks like comes from the manifest the preprocessing step wrote, which says which columns
survived, how many sequences each class has in each partition, and what is inside each
array file. Nothing is retyped from either.

Two other files are read here as well, and both belong to the single-record run this one
is measured against. Its configuration is read so that this run's configuration can
inherit from it rather than restate it, and its metrics are read so that the comparison
below quotes numbers off disk rather than numbers typed into a cell.

Two output folders, `NB06_fast` and `NB06`, so a quick check of the plumbing cannot
overwrite a result.

In [2]:
import gc
import json
import random
import textwrap
import time

import numpy as np
import pandas as pd

from baselines import mohammadi as mo
from src import inventory as inv
from src import runs as rn
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
BATCH_SIZE = int(CFG["training"]["batch_size"])
EPOCHS = int(CFG["training"]["epochs"])
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])
OUT_DIRS = {"fast": ARTIFACTS / "NB06_fast", "full": ARTIFACTS / "NB06"}
NB04_DIRS = {"fast": ARTIFACTS / "NB04_fast", "full": ARTIFACTS / "NB04"}

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(
            f"{what} not found. Looked in: {[str(p) for p in candidates]}"
        )
    return found


MANIFEST_PATH = first_existing(
    [
        REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
        NB04_DIRS["full"] / "NB04_manifest.json",
    ],
    "NB04_manifest.json",
)
PARENT_RUN = "ours_cnn_19class"
COMPARATOR_RUN = "published_cnn_19class"
PARENT_CONFIG_PATH = first_existing(
    [
        REPO_ROOT / "results" / "NB05" / PARENT_RUN / "config.json",
        ARTIFACTS / "NB05" / PARENT_RUN / "config.json",
    ],
    f"the configuration of {PARENT_RUN}",
)
COMPARATOR_METRICS_PATH = first_existing(
    [
        REPO_ROOT / "results" / "NB05" / COMPARATOR_RUN / "metrics.json",
        ARTIFACTS / "NB05" / COMPARATOR_RUN / "metrics.json",
    ],
    f"the metrics of {COMPARATOR_RUN}",
)

MANIFEST = json.loads(MANIFEST_PATH.read_text())
PARENT_SOURCE = json.loads(PARENT_CONFIG_PATH.read_text())
PUBLISHED = json.loads(COMPARATOR_METRICS_PATH.read_text())

if MANIFEST.get("is_fast_pass"):
    raise ValueError(f"{MANIFEST_PATH} came from a fast pass and is not a result")

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
SEQUENCES = {
    partition: dict(MANIFEST["arrays"][f"sequences_{partition}"]["by_class"])
    for partition in ("train", "val", "test")
}

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

print(f"seed           : {SEED}")
print(f"window, stride : {WINDOW} records, {STRIDE} records")
print(f"batch, epochs  : {BATCH_SIZE}, {EPOCHS}")
print(f"manifest       : {MANIFEST_PATH}")
print(f"  written by   : {MANIFEST['generated_by']} at {MANIFEST['git_sha']} on "
      f"{MANIFEST['generated_on']}")
print(f"parent config  : {PARENT_CONFIG_PATH}")
print(f"comparator     : {COMPARATOR_METRICS_PATH}")
print(f"features       : {len(FEATURES)}, after {', '.join(MANIFEST['columns']['dropped'])} was "
      "dropped")
print(f"classes        : {len(CLASSES)}")
print(f"arrays         : {NB04_DIRS['full']}")
print(f"fast pass to   : {OUT_DIRS['fast']}")
print(f"full pass to   : {OUT_DIRS['full']}")

assert len(FEATURES) == 44, f"expected 44 features from the manifest, got {len(FEATURES)}"
assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
assert MANIFEST["sequences"]["window"] == WINDOW and MANIFEST["sequences"]["stride"] == STRIDE, (
    "the arrays were cut at a different window or stride than config/base.yaml asks for"
)
assert MANIFEST["split"]["protocol"] == "two_tier"
assert mo.FIT["batch_size"] == BATCH_SIZE and mo.FIT["epochs"] == EPOCHS, (
    "the published training procedure and config/base.yaml disagree about batch size or "
    "epochs. The baseline is not changed to match; the config is."
)
assert PUBLISHED["labels"] == CLASSES, (
    "the comparator was scored on a different list of classes, so its per-class figures "
    "cannot be lined up with this run's"
)

seed           : 42
window, stride : 50 records, 25 records
batch, epochs  : 32, 10
manifest       : /content/repo/data/processed/NB04_manifest.json
  written by   : AG_PRAXIS_NB04_preprocessing_splits.ipynb at a557ed8 on 2026-08-04
parent config  : /content/repo/results/NB05/ours_cnn_19class/config.json
comparator     : /content/repo/results/NB05/published_cnn_19class/metrics.json
features       : 44, after Drate was dropped
classes        : 19
arrays         : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB04
fast pass to   : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06_fast
full pass to   : /content/drive/MyDrive/AG_PRAXIS_artifacts/NB06


The classes that matter most here are the ones the comparison is about, and they are not
chosen in this notebook. The rule fixed in advance is that a class counts as detected at
F1 of 0.50 or above, and the set to compare is the classes the published single-record
network scores below that line.

So the set is read off that run's metrics file rather than typed in. Six classes are
below the line. Five of them are the set. The sixth is left out because it has four test
sequences, and a per-class F1 computed on four items can only take a handful of values,
so it cannot carry a claim either way. Its figures are still reported. The cell below
checks that what it reads is what the pre-registration fixed, so if the file ever
disagreed with the plan the run would stop here rather than quietly compare a different
set.

In [3]:
DETECTED_AT = 0.50
TOO_FEW_TO_INTERPRET = "Recon-Ping_Sweep"
H2_CLASSES = [
    "Recon-VulScan",
    "Recon-OS_Scan",
    "MQTT-DDoS-Publish_Flood",
    "Spoofing",
    "MQTT-Malformed_Data",
]

below = {label: value for label, value in PUBLISHED["per_class_f1"].items() if value < DETECTED_AT}
published_table = pd.DataFrame(
    {
        "class": list(below),
        "published_f1": [below[label] for label in below],
        "test_sequences_here": [SEQUENCES["test"][label] for label in below],
        "in_the_compared_set": [label in H2_CLASSES for label in below],
    }
).sort_values("published_f1")

print(f"the comparator is {COMPARATOR_RUN}, macro F1 {PUBLISHED['macro_f1']:.4f}")
print(f"a class counts as detected at F1 {DETECTED_AT:.2f} or above")
print()
print(published_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"{len(below)} classes below the line, {len(H2_CLASSES)} of them compared.")
print(f"{TOO_FEW_TO_INTERPRET} is left out: {SEQUENCES['test'][TOO_FEW_TO_INTERPRET]} test "
      f"sequences and {SEQUENCES['val'][TOO_FEW_TO_INTERPRET]} validation sequences at this "
      "window and stride.")

assert set(below) == set(H2_CLASSES) | {TOO_FEW_TO_INTERPRET}, (
    "the classes the comparator fails are not the classes this run expects to compare"
)
assert abs(PUBLISHED["macro_f1"] - 0.7110) < 5e-5, (
    f"the comparator's macro F1 reads {PUBLISHED['macro_f1']:.4f}, not the 0.7110 this "
    "comparison is stated against"
)

the comparator is published_cnn_19class, macro F1 0.7110
a class counts as detected at F1 0.50 or above

                  class  published_f1  test_sequences_here  in_the_compared_set
          Recon-VulScan        0.0000                   18                 True
       Recon-Ping_Sweep        0.0107                    4                False
          Recon-OS_Scan        0.0343                  123                 True
MQTT-DDoS-Publish_Flood        0.1858                  215                 True
               Spoofing        0.3438                  104                 True
    MQTT-Malformed_Data        0.4883                   40                 True

6 classes below the line, 5 of them compared.
Recon-Ping_Sweep is left out: 4 test sequences and 2 validation sequences at this window and stride.


Now what the model will actually be fed.

Cutting a recording into windows of fifty records at a stride of twenty-five means each
window overlaps the one before it by half, and a window never spans two files. That turns
several million rows into a few hundred thousand windows, and it turns a small class into
a very small one: a class with a few hundred records has a few dozen windows, and one of
them ends up with two windows in validation and four in test.

That is a reporting problem rather than a modelling one, so the line is drawn here,
before anything runs. Any class with fewer than fifty sequences in a partition it is
scored on is flagged in the metrics file, and its per-class F1 is reported without being
read as a measurement of how well the class is detected. Fifty is the window length, and
it separates the three smallest classes from the rest cleanly: the next smallest class
after those three has ninety-two.

In [4]:
THIN_BELOW = 50

FLAGGED = {
    "Recon-Ping_Sweep": (
        "Too few sequences in the partitions it is scored on for an F1 to take more than a "
        "few values. Reported and not interpreted, and outside the set of classes the "
        "comparison below is scored on."
    ),
    "Recon-VulScan": (
        "Thin in every partition. Kept in the compared set, and its per-class figure carries "
        "this caveat wherever it is quoted."
    ),
    "MQTT-Malformed_Data": (
        "Thin in every partition, and the closest of the compared classes to the 0.50 line. "
        "Kept in the compared set for the same reason as the one above: dropping a borderline "
        "class after seeing its value would be choosing the result."
    ),
}

sequence_counts = pd.DataFrame(
    {
        "class": CLASSES,
        "train": [SEQUENCES["train"][c] for c in CLASSES],
        "val": [SEQUENCES["val"][c] for c in CLASSES],
        "test": [SEQUENCES["test"][c] for c in CLASSES],
        "train_records": [MANIFEST["arrays"]["records_train"]["by_class"][c] for c in CLASSES],
    }
).sort_values("test")
sequence_counts["flagged"] = sequence_counts["class"].isin(FLAGGED)

for column in ("train", "val", "test", "train_records"):
    if not pd.api.types.is_numeric_dtype(sequence_counts[column]):
        raise TypeError(f"{column} came out as {sequence_counts[column].dtype}, not numeric")

totals = {p: MANIFEST["arrays"][f"sequences_{p}"]["shape"][0] for p in ("train", "val", "test")}
print(f"sequences: train {totals['train']:,}, validation {totals['val']:,}, test "
      f"{totals['test']:,}, each {WINDOW} records of {len(FEATURES)} features")
print()
print(sequence_counts.to_string(index=False))
print()
print(f"flagged as thin, below {THIN_BELOW} in a partition it is scored on: "
      f"{', '.join(sorted(FLAGGED))}")
smallest_unflagged = sequence_counts[~sequence_counts["flagged"]][["val", "test"]].min().min()
print(f"smallest class not flagged, in either scored partition: {smallest_unflagged} sequences")

assert set(FLAGGED) <= set(CLASSES)
assert smallest_unflagged >= THIN_BELOW, (
    "a class below the line is not in the flagged list, so the caveats field would be silent "
    "about it"
)
assert sequence_counts[["train", "val", "test"]].min().min() > 0, (
    "a class has no sequences in some partition, so it cannot be trained on or scored"
)

sequences: train 249,061, validation 52,637, test 49,159, each 50 records of 44 features

                  class  train  val  test  train_records  flagged
       Recon-Ping_Sweep     24    2     4            648     True
          Recon-VulScan     86   18    18           2244     True
    MQTT-Malformed_Data    191   38    40           4813     True
 MQTT-DoS-Connect_Flood    444   92    94          11132    False
               Spoofing    497  105   104          12453    False
          Recon-OS_Scan    577  121   123          14466    False
MQTT-DDoS-Publish_Flood   1008  213   215          25227    False
 MQTT-DoS-Publish_Flood   1479  314   316          37016    False
        Recon-Port_Scan   2983  637   638          74622    False
MQTT-DDoS-Connect_Flood   6017 1286  1288         150466    False
                 Benign   6448 1379  1381         161237    False
                DoS-TCP  11471 3738  3282         286896    False
               DoS-ICMP  12403 4243  3936         31

The validation partition is loaded by nothing below. Nothing here is chosen by looking at
a score: there is one configuration, its parameters were fixed before it ran, and the test
partition is scored once. The validation counts above come from the manifest and are used
only to say which classes are thin.

The seed is set before any model is constructed. It is set again immediately before the
model that trains is built, inside the statement that fits it, so no code can run between
seeding and construction.

What that gives is a repeatable initialisation and a repeatable shuffle. It does not give
bit-identical arithmetic on a GPU, because the order a reduction accumulates in is not
fixed and floating-point addition is not associative. One seed makes this run comparable
to another run of the same thing rather than provably identical to it.

In [5]:
import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")

print(f"seeded with : {SEED}")
print(f"tensorflow  : {tf.__version__}")
print(f"keras       : {keras.__version__}")
print(f"gpu         : {[d.name for d in GPUS] if GPUS else 'none, this will be slow'}")

seeded with : 42
tensorflow  : 2.20.0
keras       : 3.13.2
gpu         : ['/physical_device:GPU:0']


The model, in two parts.

The first part is the record encoder. The published network is built exactly as it is
published and then its last layer, the softmax over the classes, is taken off. What
remains is two convolutions, two pooling layers, a flatten and a dense layer of 128, and
it turns one record into 128 numbers. That is the part this notebook does not touch. It is
applied to each of the fifty records in a window with the same weights every time, so
where a record sits in the window changes nothing about how it is read.

The second part is one LSTM layer that reads the fifty results in order and returns a
single vector, and a dense softmax on top of it. That layer is the only thing in the model
that can use the order the records came in. If it earns nothing, this is a slower way of
doing what a single-record model already does, and that is a result rather than a failure.

The cell below builds one of these and prints it, so the shapes can be read before
anything trains. It then rebuilds the published network from scratch and compares the two
encoders position by position: layer type, output shape, parameter count. What that cannot
check is that the weights are the published ones, because they are not, they are trained
here. What it checks is that the thing being trained is the published architecture with its
head removed. The model built in this cell is thrown away afterwards.

In [6]:
SPECIMEN = sq.build_model(len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS)
SPECIMEN.summary()

ENCODER_CHECK = sq.encoder_matches_baseline(SPECIMEN, len(FEATURES), len(CLASSES))
encoder_table = pd.DataFrame(ENCODER_CHECK["rows"])
print()
print("the record encoder against the published network, layer by layer")
print(encoder_table.to_string(index=False))
print()

by_layer = {layer.name: int(layer.count_params()) for layer in SPECIMEN.layers}
print(f"parameters in the encoder, one set shared by all {WINDOW} records")
print(f"  {by_layer['per_record']:>10,}")
print("parameters in the layer that reads across the window")
print(f"  {by_layer['across_the_window']:>10,}")
print("parameters in the classifier")
print(f"  {by_layer['classifier']:>10,}")
print("total")
print(f"  {SPECIMEN.count_params():>10,}")
print()
print("input shape  :", SPECIMEN.input_shape)
print("output shape :", SPECIMEN.output_shape)

assert ENCODER_CHECK["agrees"], (
    "the encoder inside this model is not the published architecture with its head removed"
)
assert SPECIMEN.input_shape == (None, WINDOW, len(FEATURES), 1)
assert SPECIMEN.output_shape == (None, len(CLASSES))

del SPECIMEN
gc.collect()

Model: "mohammadi_cnn_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ per_record (TimeDistributed)    │ (None, 50, 128)        │        80,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ across_the_window (LSTM)        │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 19)             │         2,451 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 214,227 (836.82 KB)

 Trainable params: 214,227 (836.82 KB)

 Non-trainable params: 0 (0.00 B)


the record encoder against the published network, layer by layer
 position     baseline      encoder baseline_output encoder_output  baseline_params  encoder_params
        0       Conv1D       Conv1D  (None, 42, 32) (None, 42, 32)              128             128
        1 MaxPooling1D MaxPooling1D  (None, 21, 32) (None, 21, 32)                0               0
        2       Conv1D       Conv1D  (None, 19, 64) (None, 19, 64)             6208            6208
        3 MaxPooling1D MaxPooling1D   (None, 9, 64)  (None, 9, 64)                0               0
        4      Flatten      Flatten     (None, 576)    (None, 576)                0               0
        5        Dense        Dense     (None, 128)    (None, 128)            73856           73856

parameters in the encoder, one set shared by all 50 records
      80,192
parameters in the layer that reads across the window
     131,584
parameters in the classifier
       2,451
total
     214,227

input shape  : (None, 50, 44, 1)

3772

Before any of it runs, what a run is.

A run is a configuration, and the configuration is the ledger entry. This one's parent is
the single-record convolutional run on the same split, and the parent is not retyped: it
is read from the config file that run wrote, so the two cannot drift apart.

One key differs, `model`. The split, the columns, the scaler, the optimiser, the loss, the
batch size, the number of epochs, the class weighting, the early stopping and the seed are
all the values the parent already carried.

The parent's configuration also has a key called `input`, describing the tensor its model
was given. That description is different here, and it is not a second decision: it is what
changing the model does to the shape of what goes in. Keys like that move into `observed`,
which is where this project puts things that follow from the one change rather than being
changes in their own right, and which the check ignores for that reason. The window, the
stride, the width of the LSTM and the shape of the input tensor sit there too. The window
and the stride are fixed for the whole project rather than chosen here.

`assert_single_change` compares the two configurations in both directions, so a key this
run stopped setting would count as a change as much as a key it set differently. It runs
while the configurations are built, again before the model is constructed, and again in
the last cell.

In [7]:
RUN_ID = "sequence_cnn_lstm_19class"
DESCRIPTIVE = ("input",)


def parent_config(mode):
    """The single-record run's own configuration, with the descriptive keys moved."""
    config = {k: v for k, v in PARENT_SOURCE.items() if k not in ("observed",) + DESCRIPTIVE}
    config["observed"] = {
        "mode": mode,
        "restated_from": str(PARENT_CONFIG_PATH),
        "moved_into_observed": {key: PARENT_SOURCE[key] for key in DESCRIPTIVE},
        "why_moved": (
            "the shape of the input follows from which model is being trained, so it is a "
            "consequence of the one change rather than a second change"
        ),
    }
    return config


def sequence_config(mode, parent):
    """The parent with one key changed, and everything that follows from it observed."""
    config = dict(parent)
    config["run_id"] = RUN_ID
    config["parent"] = parent["run_id"]
    config["model"] = "mohammadi_cnn_lstm"
    config["observed"] = {
        "mode": mode,
        "input": (
            f"a window of {WINDOW} records at stride {STRIDE}, reshaped to "
            f"({WINDOW}, features, 1)"
        ),
        "window": WINDOW,
        "stride": STRIDE,
        "lstm_units": sq.LSTM_UNITS,
        "n_classes": len(CLASSES),
        "classes": CLASSES,
        "features": FEATURES,
        "scaler_fitted_by": PARENT_SOURCE["observed"]["scaler_fitted_by"],
        "model": sq.describe(
            len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS
        ),
        "compared_against": COMPARATOR_RUN,
        "notebook": NOTEBOOK,
        "git_sha": GIT_SHA,
        "run_date": RUN_DATE,
    }
    config["observed"]["changed_from_parent"] = sorted(rn.assert_single_change(config, parent))
    return config


PARENT = parent_config("full")
CONFIG = sequence_config("full", PARENT)

keys = sorted(set(PARENT) | set(CONFIG))
key_table = pd.DataFrame(
    [
        {
            "key": key,
            "parent": PARENT.get(key, "-"),
            "this run": CONFIG.get(key, "-"),
            "same": PARENT.get(key) == CONFIG.get(key),
            "checked": key not in rn.IGNORED_KEYS,
        }
        for key in keys
        if key != "observed"
    ]
)
print(f"parent   : {PARENT['run_id']}, read from {PARENT_CONFIG_PATH}")
print(f"this run : {CONFIG['run_id']}")
print()
print(key_table.to_string(index=False))
print()
print(f"the one change: {', '.join(CONFIG['observed']['changed_from_parent'])}")
print(f"ignored by the check, being descriptive rather than experimental: "
      f"{', '.join(rn.IGNORED_KEYS)}")

assert CONFIG["observed"]["changed_from_parent"] == ["model"], (
    "the one change is not the model, so this run is not the experiment it says it is"
)

parent   : ours_cnn_19class, read from /content/repo/results/NB05/ours_cnn_19class/config.json
this run : sequence_cnn_lstm_19class

           key                                                                    parent                                                                  this run  same  checked
    batch_size                                                                        32                                                                        32  True     True
  class_weight                                                                      None                                                                      None  True     True
early_stopping                                                                     False                                                                     False  True     True
        epochs                                                                        10                                                                   

The metrics file carries two fields beyond the usual scores, and both are computed inside
the statement that fits the model, so a run that saved cannot be missing them.

The first is `caveats`. It lists the flagged classes with their sequence counts in each
partition, read from the manifest rather than typed in, and it also lists any class that
fell below the line and was not flagged, so a thin class nobody thought about shows up
rather than passing unnoticed. The counts describe the corpus at this window and stride,
which is what the caveat is about, and the full pass checks the arrays it loaded against
them.

The second is `comparison`. It puts this run's macro F1 and weighted F1 against the
single-record run's, and then the per-class F1 of the five compared classes side by side,
with a detected flag at 0.50 on each side. Every number on the other side is read out of
that run's metrics file.

One thing the comparison cannot do is pair the two runs item by item. They are scored on
different test partitions: one on rows of the distributed split, this one on windows of
the two-tier split. So the field holds two runs' own scores, and it says so in a line of
its own rather than leaving a reader to work it out.

In [8]:
def make_extra_metrics():
    """The caveats and the comparison, as a function of the metrics this run produced."""

    def extra(metrics):
        caveats = sq.thin_class_caveats(
            sequences=SEQUENCES, flagged=FLAGGED, thin_below=THIN_BELOW
        )
        caveats["counts_from"] = (
            f"{MANIFEST_PATH.name}, the sequence counts of the whole corpus at window "
            f"{WINDOW} and stride {STRIDE}. The full pass checks the arrays it loaded "
            "against them."
        )
        return {
            "caveats": caveats,
            "comparison": sq.comparison_against(
                metrics,
                PUBLISHED,
                run_id=COMPARATOR_RUN,
                classes=H2_CLASSES,
                threshold=DETECTED_AT,
            ),
        }

    return extra


preview = pd.DataFrame(
    {
        "class": H2_CLASSES,
        "published_f1": [PUBLISHED["per_class_f1"][c] for c in H2_CLASSES],
        "detected_published": [PUBLISHED["per_class_f1"][c] >= DETECTED_AT for c in H2_CLASSES],
        "test_sequences_here": [SEQUENCES["test"][c] for c in H2_CLASSES],
        "flagged_thin": [c in FLAGGED for c in H2_CLASSES],
    }
)
print("what the comparison will be against, per class")
print(preview.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"and overall: macro F1 {PUBLISHED['macro_f1']:.4f}, weighted F1 "
      f"{PUBLISHED['weighted_f1']:.4f}, accuracy {PUBLISHED['accuracy']:.4f} on "
      f"{PUBLISHED['n_test']:,} test rows")
print()
print("None of those five is detected at 0.50 by the single-record run, which is what puts")
print("them in this set. Whether any of them is detected here is what the run decides.")

what the comparison will be against, per class
                  class  published_f1  detected_published  test_sequences_here  flagged_thin
          Recon-VulScan        0.0000               False                   18          True
          Recon-OS_Scan        0.0343               False                  123         False
MQTT-DDoS-Publish_Flood        0.1858               False                  215         False
               Spoofing        0.3438               False                  104         False
    MQTT-Malformed_Data        0.4883               False                   40          True

and overall: macro F1 0.7110, weighted F1 0.9840, accuracy 0.9863 on 1,614,182 test rows

None of those five is detected at 0.50 by the single-record run, which is what puts
them in this set. Whether any of them is detected here is what the run decides.


The sequences themselves are read from the arrays the preprocessing step wrote, already
scaled by a scaler fitted on the training partition and nothing else. No CSV is opened
here.

The only thing the fast pass changes is how many of them are read. It takes a stratified
sample, so every class keeps the share of the sample it had of the partition, including
the classes with a handful of windows, which keep at least one. Every layer, every
setting, every check and every step is the same in both passes.

In [9]:
FAST_CAPS = {"train": 6_000, "test": 3_000}
RESUME = True
PREDICT_BATCH = 512
FIT_VERBOSE = 2


def section(title):
    print()
    print("-" * 100)
    print(title)
    print("-" * 100)


def banner(lines):
    print()
    print("#" * 100)
    for line in lines:
        print(f"#  {line[:95]:<96}#")
    print("#" * 100)


def load_sequences(fast: bool) -> dict:
    """The training and test windows, and the counts of what came back."""
    section("Loading the sequences")
    mode = "fast" if fast else "full"
    array_dir = NB04_DIRS[mode]
    fell_back = not array_dir.exists()
    if fell_back:
        array_dir = NB04_DIRS["full"]

    X, y = {}, {}
    for name in ("train", "test"):
        path = array_dir / f"sequences_{name}.npz"
        if not path.exists():
            raise FileNotFoundError(f"{path} is missing. The preprocessing step writes it.")
        with np.load(path, allow_pickle=False) as npz:
            features = [str(v) for v in npz["features"]]
            classes = [str(v) for v in npz["classes"]]
            window, stride = int(npz["window"]), int(npz["stride"])
            if features != FEATURES:
                raise ValueError(f"{path.name} holds different columns than the manifest lists")
            if classes != CLASSES:
                raise ValueError(f"{path.name} holds different classes than the manifest lists")
            if (window, stride) != (WINDOW, STRIDE):
                raise ValueError(
                    f"{path.name} was cut at window {window} stride {stride}, not {WINDOW} "
                    f"and {STRIDE}"
                )
            X[name] = npz["X"]
            y[name] = npz["y"].astype("int64")
        print(f"  {path.name:<24} {str(X[name].shape):>24}   {X[name].dtype}")

    if fell_back and fast:
        print()
        print(f"{NB04_DIRS['fast']} is not on Drive, so the fast pass takes a stratified sample")
        print("of the full arrays instead. Same windows, fewer of them, same class proportions.")
        for name in ("train", "test"):
            index = rn.stratified_subsample(y[name], cap=FAST_CAPS[name], seed=SEED)
            X[name], y[name] = X[name][index], y[name][index]
            print(f"  {name:<6} cut to {len(index):,} sequences")
        gc.collect()

    counts = {
        name: {
            label: int(n)
            for label, n in zip(CLASSES, np.bincount(y[name], minlength=len(CLASSES)))
        }
        for name in ("train", "test")
    }

    table = pd.DataFrame(
        {
            "class": CLASSES,
            "train": [counts["train"][c] for c in CLASSES],
            "test": [counts["test"][c] for c in CLASSES],
        }
    ).sort_values("test")
    print()
    print(f"read from {array_dir}")
    print(f"train {len(y['train']):,} sequences, test {len(y['test']):,}, "
          f"{WINDOW} records of {len(FEATURES)} features each, already scaled")
    print()
    print(table.to_string(index=False))

    assert X["train"].shape[1:] == (WINDOW, len(FEATURES))
    assert np.isfinite(X["train"]).all() and np.isfinite(X["test"]).all(), (
        "an array holds a value that is not finite, so nothing below this point is a result"
    )
    assert table[["train", "test"]].min().min() > 0, (
        "a class is missing from one side, so it cannot be trained on or scored"
    )
    if not fast:
        for name in ("train", "test"):
            assert counts[name] == SEQUENCES[name], (
                f"the {name} array holds different per-class counts than the manifest records, "
                "so the caveats written into the metrics would describe a different corpus"
            )

    return {
        "X_train": sq.reshape(X["train"]),
        "y_train": y["train"],
        "X_test": sq.reshape(X["test"]),
        "y_test": y["test"],
        "counts": counts,
        "array_dir": str(array_dir),
    }

The training step is one statement.

It checks the configuration against its parent, sets the seed, builds the model, fits it,
predicts, scores, and writes the five files: the configuration, the metrics, the true
labels, the predicted labels and the model. If the session ends part way through, nothing
is written, which is the point. A score that reached the screen but not the disk cannot be
checked afterwards, cannot be compared against anything and cannot go in the ledger.

The two label files hold int8 codes rather than class names, and the list those codes
index is in the metrics file under `labels`. Fifty thousand predictions are then fifty
thousand single bytes, and the names live in one place instead of being repeated on every
row.

A checkpoint is written to Drive after every epoch, so a session that ends inside the run
costs the epoch it was in rather than all ten.

In [10]:
def train(config, data, out_dir):
    """Fit and save in one statement."""
    config["observed"]["n_train"] = int(len(data["y_train"]))
    config["observed"]["n_test"] = int(len(data["y_test"]))
    config["observed"]["sequences_from"] = data["array_dir"]
    return sq.fit_and_save(
        out_dir,
        config["run_id"],
        X_train=data["X_train"],
        y_train=data["y_train"],
        X_test=data["X_test"],
        y_test=data["y_test"],
        classes=CLASSES,
        config=config,
        parent=PARENT,
        window=WINDOW,
        n_features=len(FEATURES),
        lstm_units=sq.LSTM_UNITS,
        seed=SEED,
        extra_metrics=make_extra_metrics(),
        checkpoint=True,
        predict_batch_size=PREDICT_BATCH,
        verbose=FIT_VERBOSE,
    )


def report(run):
    """The headline numbers of the run, then its per-class table."""
    metrics, config = run["metrics"], run["config"]
    comparison = metrics["comparison"]
    print(f"{config['run_id']}   {config['model']}, {config['task']}, {config['split']} split")
    print(f"  trained on {metrics['n_train']:,} sequences, tested on {metrics['n_test']:,}")
    print(f"  accuracy            {metrics['accuracy']:.4f}   "
          f"(chance {metrics['chance_rate']:.4f}, always the largest class "
          f"{metrics['majority_class_rate']:.4f})")
    print(f"  weighted P / R / F1 {metrics['weighted_precision']:.4f} / "
          f"{metrics['weighted_recall']:.4f} / {metrics['weighted_f1']:.4f}")
    print(f"  macro    P / R / F1 {metrics['macro_precision']:.4f} / "
          f"{metrics['macro_recall']:.4f} / {metrics['macro_f1']:.4f}")
    print(f"  weighted F1 minus macro F1   {metrics['weighted_f1'] - metrics['macro_f1']:.4f}")
    print(f"  macro F1 against {comparison['against']}   "
          f"{comparison['macro_f1']['this_run']:.4f} against "
          f"{comparison['macro_f1']['published']:.4f}, "
          f"{comparison['macro_f1']['difference']:+.4f}")
    print(f"  {metrics['n_parameters']:,} parameters, train "
          f"{metrics['train_seconds']:,.1f}s, predict {metrics['inference_seconds']:,.1f}s "
          f"({metrics['inference_rows_per_second']:,.0f} sequences/s)")
    print(f"  saved to {run['run_dir']}")
    print()
    print(rn.per_class_frame(metrics).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print()

Between the two passes there is a gate.

The fast pass is not there to produce a score. It is there to produce a run, and the gate
is the list of things a run has to be before the full pass is worth starting: five files
on disk, predictions stored as codes that index the label list in the metrics file, all
nineteen classes present in what was scored, a caveats field that flags the thin classes
and leaves none out, a comparison field covering the five compared classes and quoting the
right comparator, a configuration one key away from its parent, scores that are finite and
between zero and one, a loss that stayed finite through every epoch, and a saved model
that loads again from disk with the published encoder still inside it.

If any of those fails, the full pass does not start. The cost of finding out is a couple
of minutes instead of an hour, and the same gate runs again on the full pass afterwards.

In [11]:
def gate(run, *, mode) -> pd.DataFrame:
    """Every check a run has to pass. Raises if any of them fails."""
    section(f"The gate, on the {mode} pass")
    run_dir = Path(run["run_dir"])
    metrics, config = run["metrics"], run["config"]
    checks = []

    def check(name, ok, detail):
        checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})

    files = sorted(p.name for p in run_dir.iterdir())
    wanted = ["config.json", "metrics.json", "model.keras", "y_pred.npy", "y_true.npy"]
    check("the five files are on disk", all(f in files for f in wanted), ", ".join(files))

    y_true = np.load(run_dir / "y_true.npy")
    y_pred = np.load(run_dir / "y_pred.npy")
    check(
        "labels are stored as int8 codes",
        y_true.dtype == np.int8 and y_pred.dtype == np.int8,
        f"y_true {y_true.dtype}, y_pred {y_pred.dtype}",
    )
    check(
        "the codes index the label list in metrics.json",
        metrics["labels"] == CLASSES
        and int(y_true.min()) >= 0
        and int(max(y_true.max(), y_pred.max())) < len(CLASSES),
        f"{len(metrics['labels'])} labels, codes {int(y_true.min())} to "
        f"{int(max(y_true.max(), y_pred.max()))}",
    )
    check(
        "all 19 classes were scored",
        len(set(y_true.tolist())) == len(CLASSES) and len(metrics["per_class_f1"]) == len(CLASSES),
        f"{len(set(y_true.tolist()))} classes in the test partition, "
        f"{len(metrics['per_class_f1'])} per-class scores",
    )

    scores = [metrics["accuracy"], metrics["weighted_f1"], metrics["macro_f1"]]
    check(
        "accuracy, weighted F1 and macro F1 are finite and between 0 and 1",
        all(np.isfinite(v) and 0.0 <= v <= 1.0 for v in scores),
        f"accuracy {metrics['accuracy']:.4f}, weighted {metrics['weighted_f1']:.4f}, "
        f"macro {metrics['macro_f1']:.4f}",
    )
    losses = metrics.get("history", {}).get("loss", [])
    check(
        "the loss stayed finite through every epoch",
        bool(losses) and all(np.isfinite(v) for v in losses),
        f"{len(losses)} epochs, last loss {losses[-1]:.4f}" if losses else "no history",
    )

    caveats = metrics["caveats"]
    check(
        "the caveats field flags the thin classes and leaves none out",
        sorted(entry["label"] for entry in caveats["flagged"]) == sorted(FLAGGED)
        and not caveats["unflagged_and_thin"],
        f"flagged {', '.join(sorted(e['label'] for e in caveats['flagged']))}; "
        f"unflagged and thin {caveats['unflagged_and_thin'] or 'none'}",
    )
    comparison = metrics["comparison"]
    check(
        "the comparison covers the five classes, against the right run",
        sorted(comparison["per_class_f1"]) == sorted(H2_CLASSES)
        and comparison["against"] == COMPARATOR_RUN
        and abs(comparison["macro_f1"]["published"] - 0.7110) < 5e-5,
        f"{comparison['n_classes_compared']} classes against {comparison['against']} at "
        f"macro F1 {comparison['macro_f1']['published']:.4f}",
    )

    changed = sorted(rn.assert_single_change(config, PARENT))
    check(
        "the configuration is one key from its parent, and the key is the model",
        changed == ["model"],
        f"{config['parent']} to {config['run_id']}, changed "
        f"{', '.join(changed) if changed else 'nothing'}",
    )

    saved = keras.saving.load_model(run_dir / "model.keras")
    encoder = sq.encoder_matches_baseline(saved, len(FEATURES), len(CLASSES))
    check(
        "the saved model loads and still holds the published encoder",
        encoder["agrees"] and saved.input_shape == (None, WINDOW, len(FEATURES), 1),
        f"{saved.count_params():,} parameters, encoder {encoder['encoder_params']:,}, "
        f"input {saved.input_shape}",
    )
    del saved
    gc.collect()

    table = pd.DataFrame(checks)
    print(table.to_string(index=False))
    failed = table[table["result"] == "FAIL"]
    print()
    print(f"{len(table) - len(failed)} of {len(table)} passed")
    if len(failed):
        raise AssertionError(
            f"the {mode} pass failed {len(failed)} check(s): "
            f"{', '.join(failed['check'].tolist())}. The full pass does not start."
        )
    return table


def run_pass(fast: bool) -> dict:
    mode = "fast" if fast else "full"
    out_dir = OUT_DIRS[mode]
    out_dir.mkdir(parents=True, exist_ok=True)
    config = sequence_config(mode, PARENT)

    started = time.time()
    existing = sq.load_run(out_dir, RUN_ID) if RESUME else None
    if existing is not None:
        section(f"{RUN_ID} is already complete in {out_dir}")
        print(f"read back from {existing['run_dir']}, macro F1 "
              f"{existing['metrics']['macro_f1']:.4f}")
        run, refitted = existing, False
    else:
        data = load_sequences(fast)
        section(f"Training - {mode} pass")
        run = train(config, data, out_dir)
        refitted = True
        data.clear()
        gc.collect()
        print()

    report(run)
    checks = gate(run, mode=mode)
    return {
        "mode": mode,
        "run": run,
        "out_dir": out_dir,
        "checks": checks,
        "refitted": refitted,
        "elapsed_s": time.time() - started,
    }

Before starting it, how long it will take.

The arithmetic is not a measurement. It counts the gradient steps the full pass will
actually take, which is exact, and divides by a guessed rate, which is not. Each step here
puts thirty-two windows through the encoder, and each window is fifty records, so a step
does fifty times the convolution work a single-record step does while there are
twenty-five times fewer of them.

The rate is written in the cell and can be changed once a real run has been timed. Its
only job is to say whether this is a twenty minute job or most of a day, so the decision
to come back with a longer session gets made now rather than an hour in.

In [12]:
STEP_RATE = 100
LOAD_MB_PER_SECOND = 60


def estimate(mode: str) -> dict:
    array_dir = NB04_DIRS[mode] if NB04_DIRS[mode].exists() else NB04_DIRS["full"]
    fell_back = array_dir != NB04_DIRS[mode]
    n_train = MANIFEST["arrays"]["sequences_train"]["shape"][0]
    n_test = MANIFEST["arrays"]["sequences_test"]["shape"][0]
    if mode == "fast" and fell_back:
        n_train, n_test = min(n_train, FAST_CAPS["train"]), min(n_test, FAST_CAPS["test"])

    megabytes = sum(
        entry["bytes"] / 1e6
        for entry in MANIFEST["files"]
        if entry["file"] in ("sequences_train.npz", "sequences_test.npz")
    )
    steps = int(np.ceil(n_train / BATCH_SIZE)) * EPOCHS
    done = RESUME and rn.load_run(OUT_DIRS[mode], RUN_ID) is not None
    minutes = 0.0 if done else (megabytes / LOAD_MB_PER_SECOND + steps / STEP_RATE) / 60
    return {
        "pass": mode,
        "train_sequences": n_train,
        "test_sequences": n_test,
        "gradient_steps": steps,
        "minutes": minutes,
        "status": "already complete" if done else "to run",
    }


print(f"assumed rates: {STEP_RATE} gradient steps a second, {LOAD_MB_PER_SECOND} MB a second "
      "off Drive.")
print("Guesses, not measurements. The step counts are exact.")
print()
ESTIMATE = pd.DataFrame([estimate(mode) for mode in ("fast", "full")])
print(ESTIMATE.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
print()
print(f"total {ESTIMATE['minutes'].sum():,.1f} minutes "
      f"({ESTIMATE['minutes'].sum() / 60:,.1f} hours)")
print()
print("If that is longer than the session available, stop here. A finished run is read back")
print("off disk rather than fitted again, so two short sessions cost one extra load of the")
print("arrays and nothing else.")

assumed rates: 100 gradient steps a second, 60 MB a second off Drive.
Guesses, not measurements. The step counts are exact.

pass  train_sequences  test_sequences  gradient_steps  minutes status
fast           249061           49159           77840     13.7 to run
full           249061           49159           77840     13.7 to run

total 27.4 minutes (0.5 hours)

If that is longer than the session available, stop here. A finished run is read back
off disk rather than fitted again, so two short sessions cost one extra load of the
arrays and nothing else.


Both passes run here, fast first, then the gate, then the full pass. A failure anywhere in
the fast pass or in the gate stops the cell before the long run starts.

Re-running this cell reads a finished run back off disk instead of fitting it again.
Deleting a run's folder makes it run again.

In [13]:
banner([
    "FAST PASS",
    "the same model, the same settings, the same checks, on a sample of the windows",
    "for shapes and plumbing only",
    "not a result, and never entered in the ledger",
])
FAST = run_pass(True)

banner([
    "FULL PASS",
    f"every window of the two-tier split, {EPOCHS} epochs at batch {BATCH_SIZE}",
    "this is the pass that goes in the ledger",
])
FULL = run_pass(False)

banner([
    f"fast pass {FAST['elapsed_s'] / 60:.1f} min, full pass {FULL['elapsed_s'] / 60:.1f} min",
    f"{FULL['run']['run_dir']}",
    "refitted here" if FULL["refitted"] else "read back from an earlier session",
])


####################################################################################################
#  FAST PASS                                                                                       #
#  the same model, the same settings, the same checks, on a sample of the windows                  #
#  for shapes and plumbing only                                                                    #
#  not a result, and never entered in the ledger                                                   #
####################################################################################################

----------------------------------------------------------------------------------------------------
Loading the sequences
----------------------------------------------------------------------------------------------------
  sequences_train.npz                 (181, 50, 44)   float32
  sequences_test.npz                   (80, 50, 44)   float32

read from /content/drive/MyDrive/AG_PRAXIS_

The result, without any averaging first.

Every class, its F1, and how many test sequences that F1 was computed on, sorted so the
classes that fail are at the top. A class sitting at the top of this table is a class the
model does not detect, and no headline number above it changes that. The last column
carries the caveat: three of these rows rest on too few sequences to read as measurements.

In [14]:
METRICS = FULL["run"]["metrics"]
COMPARISON = METRICS["comparison"]

per_class = rn.per_class_frame(METRICS)
per_class["thin"] = per_class["label"].isin(FLAGGED)
per_class["compared"] = per_class["label"].isin(H2_CLASSES)

print("=" * 100)
print(f"{RUN_ID} - per-class F1 on all {len(CLASSES)} classes")
print("=" * 100)
print(per_class.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"accuracy {METRICS['accuracy']:.4f}, weighted F1 {METRICS['weighted_f1']:.4f}, "
      f"macro F1 {METRICS['macro_f1']:.4f}")
print(f"weighted F1 minus macro F1 {METRICS['weighted_f1'] - METRICS['macro_f1']:.4f}")
print()
readable = per_class[~per_class["thin"]]
print(f"classes at F1 0.50 or above: {int((per_class['f1'] >= DETECTED_AT).sum())} of "
      f"{len(per_class)}, or {int((readable['f1'] >= DETECTED_AT).sum())} of {len(readable)} "
      "leaving out the three thin ones")
print(f"classes at F1 0.00        : "
      f"{', '.join(per_class.loc[per_class['f1'] == 0.0, 'label']) or 'none'}")

sequence_cnn_lstm_19class - per-class F1 on all 19 classes
                  label  precision  recall     f1  support  thin  compared
       Recon-Ping_Sweep     0.0000  0.0000 0.0000        4  True     False
MQTT-DDoS-Publish_Flood     0.8182  0.0419 0.0796      215 False      True
               DoS-ICMP     0.4790  0.2378 0.3178     3936 False     False
                DoS-TCP     0.5833  0.4628 0.5161     3282 False     False
          Recon-VulScan     0.8750  0.3889 0.5385       18  True      True
          Recon-OS_Scan     0.8000  0.4878 0.6061      123 False      True
 MQTT-DoS-Publish_Flood     0.6065  1.0000 0.7551      316 False     False
               Spoofing     0.8152  0.7212 0.7653      104 False      True
              DDoS-ICMP     0.6939  0.8700 0.7721     7826 False     False
                DoS-SYN     0.8465  0.7626 0.8023     3942 False     False
               DDoS-TCP     0.7796  0.8510 0.8137     7302 False     False
    MQTT-Malformed_Data     0.8889  0.800

Then the comparison the run was built for.

The same five classes, the single-record network's F1 on each and this model's, and
whether each one clears 0.50 on either side. Underneath, the two macro figures. The two
runs are scored on different test partitions, so these are two runs' own scores rather
than a paired comparison, and the count that matters is how many of the five cross the
line here that did not cross it there.

In [15]:
compare_table = pd.DataFrame(
    [
        {
            "class": label,
            "published_f1": values["published"],
            "sequence_f1": values["this_run"],
            "difference": values["difference"],
            "detected_published": values["detected_published"],
            "detected_here": values["detected_this_run"],
            "test_sequences": METRICS["support"][label],
            "thin": label in FLAGGED,
        }
        for label, values in COMPARISON["per_class_f1"].items()
    ]
).sort_values("sequence_f1", ascending=False)

for column in ("published_f1", "sequence_f1", "difference", "test_sequences"):
    if not pd.api.types.is_numeric_dtype(compare_table[column]):
        raise TypeError(f"{column} came out as {compare_table[column].dtype}, not numeric")

print("=" * 100)
print(f"the five classes {COMPARATOR_RUN} does not detect, at a threshold of {DETECTED_AT:.2f}")
print("=" * 100)
print(compare_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"detected there : {COMPARISON['n_detected_published']} of "
      f"{COMPARISON['n_classes_compared']}")
print(f"detected here  : {COMPARISON['n_detected_this_run']} of "
      f"{COMPARISON['n_classes_compared']}"
      + (f", newly: {', '.join(COMPARISON['newly_detected'])}"
         if COMPARISON["newly_detected"] else ""))
print()
print(f"macro F1    {COMPARISON['macro_f1']['this_run']:.4f} here against "
      f"{COMPARISON['macro_f1']['published']:.4f} there, "
      f"{COMPARISON['macro_f1']['difference']:+.4f}")
print(f"weighted F1 {COMPARISON['weighted_f1']['this_run']:.4f} here against "
      f"{COMPARISON['weighted_f1']['published']:.4f} there, "
      f"{COMPARISON['weighted_f1']['difference']:+.4f}")
print()
print(textwrap.fill(COMPARISON["test_partitions_differ"], width=98))
print()
print(textwrap.fill(METRICS["caveats"]["reading"], width=98))
for entry in METRICS["caveats"]["flagged"]:
    counts = entry["sequences"]
    print(f"  {entry['label']:<20} train {counts['train']:>6,}  val {counts['val']:>5,}  "
          f"test {counts['test']:>5,}")

the five classes published_cnn_19class does not detect, at a threshold of 0.50
                  class  published_f1  sequence_f1  difference  detected_published  detected_here  test_sequences  thin
    MQTT-Malformed_Data        0.4883       0.8421      0.3538               False           True              40  True
               Spoofing        0.3438       0.7653      0.4215               False           True             104 False
          Recon-OS_Scan        0.0343       0.6061      0.5718               False           True             123 False
          Recon-VulScan        0.0000       0.5385      0.5385               False           True              18  True
MQTT-DDoS-Publish_Flood        0.1858       0.0796     -0.1061               False          False             215 False

detected there : 0 of 5
detected here  : 4 of 5, newly: MQTT-Malformed_Data, Recon-OS_Scan, Recon-VulScan, Spoofing

macro F1    0.7138 here against 0.7110 there, +0.0028
weighted F1 0.8064 here again

The ledger entry, ready to paste into `RESULTS_LEDGER.md`. The check that this run is one
key from its parent runs once more here, on the configuration that was actually written to
disk, so the entry cannot claim a single change that the saved file does not support.

Colab cannot push to the repository from a cell, so until the artefacts are moved across by
hand the saved copy of this notebook is the only durable record of what the run produced.

In [16]:
SAVED_CONFIG = json.loads((Path(FULL["run"]["run_dir"]) / "config.json").read_text())
CHANGED = sorted(rn.assert_single_change(SAVED_CONFIG, PARENT))
print(f"config.json on disk against {PARENT['run_id']}: "
      f"{', '.join(CHANGED) if CHANGED else 'no key differs'}")
assert CHANGED == ["model"], "the saved configuration is not one key from its parent"

STATUS = "reference run" + (", working tree dirty" if GIT_DIRTY else "")
weakest = sorted(METRICS["per_class_f1"].items(), key=lambda pair: pair[1])[:4]
zero = sorted(label for label, value in METRICS["per_class_f1"].items() if value == 0.0)
thin_line = "; ".join(
    f"{entry['label']} ({entry['sequences']['test']} test, {entry['sequences']['val']} validation)"
    for entry in METRICS["caveats"]["flagged"]
)
compared_line = ", ".join(
    f"{row['class']} {row.published_f1:.4f} to {row.sequence_f1:.4f}"
    for _, row in compare_table.iterrows()
)

entry = f"""
### NB06 — {SAVED_CONFIG["run_id"]} ({RUN_DATE})

| field | value |
|---|---|
| notebook | {NOTEBOOK} |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SAVED_CONFIG["seed"]} |
| pass reported | full |
| runtime | fast {FAST["elapsed_s"] / 60:.1f} min, full {FULL["elapsed_s"] / 60:.1f} min |
| model | {SAVED_CONFIG["model"]}, the published encoder per record and one LSTM of {sq.LSTM_UNITS} across the window |
| parameters | {METRICS["n_parameters"]:,} |
| task | {SAVED_CONFIG["task"]}, {METRICS["n_classes"]} classes |
| split | {SAVED_CONFIG["split"]} |
| input | {WINDOW} records at stride {STRIDE}, {len(FEATURES)} features, no column excluded |
| parent | {SAVED_CONFIG["parent"]} |
| the one change | {", ".join(CHANGED)} |
| training sequences | {METRICS["n_train"]:,} |
| test sequences | {METRICS["n_test"]:,} |
| accuracy | {METRICS["accuracy"]:.4f} (chance {METRICS["chance_rate"]:.4f}, largest class {METRICS["majority_class_rate"]:.4f}) |
| weighted P / R / F1 | {METRICS["weighted_precision"]:.4f} / {METRICS["weighted_recall"]:.4f} / {METRICS["weighted_f1"]:.4f} |
| macro P / R / F1 | {METRICS["macro_precision"]:.4f} / {METRICS["macro_recall"]:.4f} / {METRICS["macro_f1"]:.4f} |
| weighted F1 minus macro F1 | {METRICS["weighted_f1"] - METRICS["macro_f1"]:.4f} |
| macro F1 against {COMPARATOR_RUN} | {COMPARISON["macro_f1"]["this_run"]:.4f} against {COMPARISON["macro_f1"]["published"]:.4f}, {COMPARISON["macro_f1"]["difference"]:+.4f} |
| the five compared classes, F1 | {compared_line} |
| detected at F1 {DETECTED_AT:.2f} | {COMPARISON["n_detected_this_run"]} of {COMPARISON["n_classes_compared"]} here, {COMPARISON["n_detected_published"]} of {COMPARISON["n_classes_compared"]} there{", newly: " + ", ".join(COMPARISON["newly_detected"]) if COMPARISON["newly_detected"] else ""} |
| classes at F1 0.00 | {", ".join(zero) if zero else "none"} |
| four weakest classes | {", ".join(f"{label} {value:.2f}" for label, value in weakest)} |
| too few sequences to interpret | {thin_line} |
| gate | {len(FULL["checks"])} checks, all passed |
| train seconds | {METRICS["train_seconds"]:,.1f} |
| inference seconds | {METRICS["inference_seconds"]:,.1f} ({METRICS["inference_rows_per_second"]:,.0f} sequences/s) |
| artifacts | {FULL["run"]["run_dir"]} |
| status | {STATUS} |

| class | F1 | test sequences | thin |
|---|---|---|---|
""" + "\n".join(
    f"| {row.label} | {row.f1:.4f} | {int(row.support):,} | {'yes' if row.thin else ''} |"
    for row in per_class.itertuples()
)

print("=" * 100)
print("paste into RESULTS_LEDGER.md")
print("=" * 100)
print(entry)

config.json on disk against ours_cnn_19class: model
paste into RESULTS_LEDGER.md

### NB06 — sequence_cnn_lstm_19class (2026-08-07)

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB06_sequence_model.ipynb |
| run date | 2026-08-07 |
| git sha | 20d4709 |
| seed | 42 |
| pass reported | full |
| runtime | fast 0.4 min, full 41.0 min |
| model | mohammadi_cnn_lstm, the published encoder per record and one LSTM of 128 across the window |
| parameters | 214,227 |
| task | 19-class, 19 classes |
| split | two_tier |
| input | 50 records at stride 25, 44 features, no column excluded |
| parent | ours_cnn_19class |
| the one change | model |
| training sequences | 249,061 |
| test sequences | 49,159 |
| accuracy | 0.8197 (chance 0.0526, largest class 0.1592) |
| weighted P / R / F1 | 0.8087 / 0.8197 / 0.8064 |
| macro P / R / F1 | 0.7813 / 0.7093 / 0.7138 |
| weighted F1 minus macro F1 | 0.0926 |
| macro F1 against published_cnn_19class | 0.7138 against 0.7110, +0.0028 |
| the five compa